In [1]:
import kagglehub
path = kagglehub.dataset_download("kostasdiamantaras/chest-xrays-bacterial-viral-pneumonia-normal")

100%|██████████| 1.14G/1.14G [00:19<00:00, 63.4MB/s]

Extracting files...


In [2]:
import os

print("Dataset path:", path)
print("\nFiles/folders inside:")
print(os.listdir(path))

Dataset path: /root/.cache/kagglehub/datasets/kostasdiamantaras/chest-xrays-bacterial-viral-pneumonia-normal/versions/1

Files/folders inside:
['train_images', 'test_images', 'labels_train.csv']


In [3]:
import pandas as pd

labels = pd.read_csv(os.path.join(path, "labels_train.csv"))

print(labels.head())
print("\nColumns:")
print(labels.columns)

print("\nNumber of images:", len(labels))

                     file_name  class_id
0  img_4358977458434011046.jpg         1
1  img_5224016757187192130.jpg         2
2  img_3065202206106254707.jpg         2
3  img_6304894865561547174.jpg         1
4  img_3371338542810939877.jpg         2

Columns:
Index(['file_name', 'class_id'], dtype='object')

Number of images: 4672


In [4]:
print(labels["class_id"].value_counts().sort_index())

class_id
0    1227
1    2238
2    1207
Name: count, dtype: int64


In [5]:
print("\nClass mapping:")
print("0 = Normal")
print("1 = Bacterial Pneumonia")
print("2 = Viral Pneumonia")


Class mapping:
0 = Normal
1 = Bacterial Pneumonia
2 = Viral Pneumonia


In [6]:
class_counts = labels["class_id"].value_counts().sort_index()

print("Class distribution:")
print("Normal:", class_counts.get(0, 0))
print("Bacterial Pneumonia:", class_counts.get(1, 0))
print("Viral Pneumonia:", class_counts.get(2, 0))

Class distribution:
Normal: 1227
Bacterial Pneumonia: 2238
Viral Pneumonia: 1207


In [7]:
import os

print("Dataset path:")
print(path)

print("\nContents of dataset:")
for item in os.listdir(path):
    print(item)

print("\nTrain folder contents:")
train_path = os.path.join(path, "train_images")

if os.path.exists(train_path):
    print(os.listdir(train_path)[:20])
else:
    print("train_images folder does NOT exist")

Dataset path:
/root/.cache/kagglehub/datasets/kostasdiamantaras/chest-xrays-bacterial-viral-pneumonia-normal/versions/1

Contents of dataset:
train_images
test_images
labels_train.csv

Train folder contents:
['train_images']


In [8]:
import os
import pandas as pd
import numpy as np
from PIL import Image

labels_df = pd.read_csv(os.path.join(path, "labels_train.csv"))

train_images_path = os.path.join(path, "train_images", "train_images")

X = []
y = []

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

import kagglehub


if 'path' not in globals():
    path = kagglehub.dataset_download(
        "kostasdiamantaras/chest-xrays-bacterial-viral-pneumonia-normal"
    )
print("Dataset path:", path)

labels_df = pd.read_csv(os.path.join(path, "labels_train.csv"))

train_images_path = os.path.join(path, "train_images", "train_images")

class_names = ["Normal", "Bacterial Pneumonia", "Viral Pneumonia"]


X = []
y = []
skipped = 0

for index, row in labels_df.iterrows():
    file_name = row["file_name"]
    class_id = row["class_id"]
    img_path = os.path.join(train_images_path, file_name)

    try:

        img = Image.open(img_path).convert("L")
        img = img.resize((128, 128))


        img_array = np.array(img) / 255.0

        X.append(img_array)
        y.append(class_id)
    except Exception as e:

        skipped += 1
        continue

print(f"Loaded {len(X)} images successfully. Skipped {skipped} bad/missing images.")

# Convert to NumPy arrays
X = np.array(X)
y = np.array(y)

# Add channel dimension -> (num_images, 128, 128, 1) for CNN input
X = X.reshape(-1, 128, 128, 1)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

# ---------------------------
# 5. Train/validation split (stratified)
# ---------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])

# ---------------------------
# 6. Handle class imbalance using class weights
# ---------------------------
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights_array))
print("Class weights:", class_weights)

# ---------------------------
# 7. Build a simple CNN
# ---------------------------
model = models.Sequential([
    layers.Input(shape=(128, 128, 1)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(3, activation='softmax')  # 3 classes
])

# ---------------------------
# 8. Compile the model
# ---------------------------
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ---------------------------
# 9. Train the model
# ---------------------------
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stop]
)

# ---------------------------
# 10. Plot training vs validation accuracy & loss
# ---------------------------
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# ---------------------------
# 11. Evaluate on validation set
# ---------------------------
val_loss, val_accuracy = model.evaluate(X_val, y_val)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

# ---------------------------
# 12. Generate predictions
# ---------------------------
y_pred_probs = model.predict(X_val)
y_pred = np.argmax(y_pred_probs, axis=1)

# ---------------------------
# 13. Confusion matrix
# ---------------------------
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
plt.title("Confusion Matrix")
plt.show()

# ---------------------------
# 14. Classification report
# ---------------------------
print("Classification Report:\n")
print(classification_report(y_val, y_pred, target_names=class_names))

# ---------------------------
# 15. Display sample predictions
# ---------------------------
plt.figure(figsize=(12, 8))
sample_indices = np.random.choice(len(X_val), 6, replace=False)

for i, idx in enumerate(sample_indices):
    plt.subplot(2, 3, i + 1)
    plt.imshow(X_val[idx].reshape(128, 128), cmap='gray')
    actual = class_names[y_val[idx]]
    predicted = class_names[y_pred[idx]]
    plt.title(f"Actual: {actual}\nPredicted: {predicted}", fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()

# ---------------------------
# 16. Final summary
# ---------------------------
print("=" * 50)
print("Project completed successfully")
print("Classes: Normal, Bacterial Pneumonia, Viral Pneumonia")
print(f"Final Validation Accuracy: {val_accuracy:.4f}")
print("=" * 50)

Dataset path: /root/.cache/kagglehub/datasets/kostasdiamantaras/chest-xrays-bacterial-viral-pneumonia-normal/versions/1
Loaded 4672 images successfully. Skipped 0 bad/missing images.
Shape of X: (4672, 128, 128, 1)
Shape of y: (4672,)
Training samples: 3737
Validation samples: 935
Class weights: {0: np.float64(1.2697927285083248), 1: np.float64(0.695903165735568), 2: np.float64(1.2895100069013112)}


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,451 (12.61 MB)

 Trainable params: 3,304,451 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
117/117 ━━━━━━━━━━━━━━━━━━━━ 114s 954ms/step - accuracy: 0.5999 - loss: 0.7974 - val_accuracy: 0.7701 - val_loss: 0.5644
Epoch 2/15
117/117 ━━━━━━━━━━━━━━━━━━━━ 109s 933ms/step - accuracy: 0.7378 - loss: 0.5764 - val_accuracy: 0.7786 - val_loss: 0.5458
Epoch 3/15
117/117 ━━━━━━━━━━━━━━━━━━━━ 115s 982ms/step - accuracy: 0.7680 - loss: 0.5411 - val_accuracy: 0.7797 - val_loss: 0.5417
Epoch 4/15
 85/117 ━━━━━━━━━━━━━━━━━━━━ 28s 879ms/step - accuracy: 0.7563 - loss: 0.5270

In [ ]:

from google.colab import files
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import io

# Class names in the same order as class_id (0, 1, 2)
class_names = ["Normal", "Bacterial Pneumonia", "Viral Pneumonia"]

# ---------------------------
# 2. Upload an image from your computer
# ---------------------------
uploaded = files.upload()  # Opens a file picker; upload exactly one image

# Get the uploaded file's name and raw bytes
file_name = list(uploaded.keys())[0]
file_bytes = uploaded[file_name]

# ---------------------------
# 3. Try to read and process the uploaded file
# ---------------------------
try:
    # Open the image from the uploaded bytes
    original_img = Image.open(io.BytesIO(file_bytes))

    # Display the original uploaded image
    plt.figure(figsize=(4, 4))
    plt.imshow(original_img)
    plt.title("Uploaded Image")
    plt.axis('off')
    plt.show()

    # ---------------------------
    # 4. Convert to grayscale (same as training)
    # ---------------------------
    gray_img = original_img.convert("L")

    # ---------------------------
    # 5. Resize to 128x128 (same as training)
    # ---------------------------
    resized_img = gray_img.resize((128, 128))

    # ---------------------------
    # 6. Normalize pixel values to 0-1 (same as training)
    # ---------------------------
    img_array = np.array(resized_img) / 255.0

    # ---------------------------
    # 7. Reshape for CNN input -> (1, 128, 128, 1)
    # ---------------------------
    input_array = img_array.reshape(1, 128, 128, 1)

    # ---------------------------
    # 8. Predict using the already-trained model
    # ---------------------------
    prediction = model.predict(input_array)

    # ---------------------------
    # 9. Find the predicted class (highest probability)
    # ---------------------------
    predicted_class_id = np.argmax(prediction)
    predicted_class_name = class_names[predicted_class_id]
    confidence = prediction[0][predicted_class_id] * 100

    # ---------------------------
    # 11. Display the image with prediction as title
    # ---------------------------
    plt.figure(figsize=(4, 4))
    plt.imshow(resized_img, cmap='gray')
    plt.title(f"Prediction: {predicted_class_name} ({confidence:.2f}%)")
    plt.axis('off')
    plt.show()

    # ---------------------------
    # 12. Print predicted class and confidence
    # ---------------------------
    print("Predicted Class:", predicted_class_name)
    print(f"Confidence: {confidence:.2f}%")

    # ---------------------------
    # 13. Print probabilities for all three classes
    # ---------------------------
    print("\nClass Probabilities:")
    for i, class_name in enumerate(class_names):
        prob = prediction[0][i] * 100
        print(f"{class_name}: {prob:.2f}%")

# ---------------------------
# 14. Handle the case where the uploaded file is not a valid image
# ---------------------------
except Exception as e:
    print("Error: The uploaded file could not be read as an image.")
    print("Please upload a valid image file (e.g., .jpg, .png) and try again.")